In [1]:
from ngsolve import *
from netgen.geom2d import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np

# ------------------------------------------------------------
# Parameter
# ------------------------------------------------------------
H, L = 4.0, 28.0
cx, cy, R = 7.0, 0.0, 0.5

nu = 1e-3       #viscosity

# ------------------------------------------------------------
# Geometrie
# Rectangle(width, height) liegt standardmäßig in [0,L]×[0,H]
# → daher zuerst erzeugen, dann verschieben
# ------------------------------------------------------------
rect = MoveTo(0,-H/2).Rectangle(L, H).Face()

rect.edges.Min(X).name = "inlet"
rect.edges.Max(X).name = "outlet"
rect.edges.Min(Y).name = "walls"
rect.edges.Max(Y).name = "walls"

cyl = Circle((cx,cy), R).Face()
cyl.edges.name = "obstacle"

shape = rect - cyl
#Draw(shape)

# ------------------------------------------------------------
# Mesh
# ------------------------------------------------------------
mesh = Mesh(OCCGeometry(shape,dim=2).GenerateMesh(maxh=0.4))
mesh.Refine()
Draw(mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [2]:
# ------------------------------------------------------------
# FE-Räume (Taylor–Hood)
# ------------------------------------------------------------
V = VectorH1(mesh, order=2, dirichlet="inlet|walls|obstacle")
Q = H1(mesh, order=1)   # Variante C: integral(p)=0

X = FESpace([V, Q], constraints=[])
(u, p) = X.TrialFunction()
(v, q) = X.TestFunction()

gfu = GridFunction(X)
gfu_u, gfu_p = gfu.components


In [3]:
# ------------------------------------------------------------
# Inlet-Profil (parabolisch auf [-H/2, H/2])            -> Randbedingungen bei Inlet:  u = uin bei "inlet"
# ------------------------------------------------------------
Umax = 0.1
uin_x = Umax*(1 - (2*y/H)**2)   # Maximum bei y=0
uin = CoefficientFunction((uin_x, 0))

gfu_u.Set(uin, definedon=mesh.Boundaries("inlet"))


In [4]:
# ------------------------------------------------------------
# Stokes: schwache Form
# ------------------------------------------------------------

#für stabilität
alpha = 1e-10
gamma = 1e-2


a = BilinearForm(X)
a += nu*InnerProduct(Grad(u), Grad(v)) * dx         #TODO Sym weglassen
a += gamma * div(u) * div(v) * dx
a += -div(v)*p * dx
a += -div(u)*q * dx
a += alpha * p*q * dx

L = LinearForm(X)   # keine Volumenkräfte

a.Assemble()
L.Assemble()

# ------------------------------------------------------------
# Lösen -- hatte starke Probleme beim lösen
# ------------------------------------------------------------

inv_stokes = a.mat.Inverse(X.FreeDofs())                #TODO

res = L.vec - a.mat*gfu.vec
gfu.vec.data += inv_stokes * res

Draw (gfu.components[0], mesh);
# ------------------------------------------------------------
# Visualisierung
# ------------------------------------------------------------
Draw(gfu_u, mesh, "velocity")
Draw(gfu_p, mesh, "pressure")


# Stokes hier fertig gelöst -> jetzt gehts an NavierStokes


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [14]:
# 'dt = 0.05
# t_end = 20.0
# t = 0.0

# # -------------------------------
# # Massenmatrix (nur Geschwindigkeitsteil, aber im Gesamtraum)
# # -------------------------------
# m = BilinearForm(X)
# m += InnerProduct(u, v) * dx
# m.Assemble()

# # -------------------------------
# # Linke Seite: impliziter Euler + Stokes
# # -------------------------------
# A_dt = BilinearForm(X)

# A_dt += (1/dt) * InnerProduct(u, v) * dx
# A_dt += nu * InnerProduct(Grad(u), Grad(v)) * dx
# A_dt += gamma * div(u)*div(v) * dx
# A_dt += -div(v)*p * dx
# A_dt += -div(u)*q * dx
# A_dt += alpha * p*q * dx

# A_dt.Assemble()

# invA_dt = A_dt.mat.Inverse(X.FreeDofs())

# # Platzhalter für u^n als CoefficientFunction
# # gfu_old enthält u^n

# gfu_old = GridFunction(X)
# gfu_old.vec.data = gfu.vec    # Startwert = Stokes

# u_old = gfu_old.components[0]   # DAS ist der Koeffizient

# # Konvektionsform (explizit!)
# conv_form = BilinearForm(X, nonassemble=True)
# conv_form += -InnerProduct(Grad(u_old) * u_old, v) * dx

# #gfu_old = GridFunction(X)
# #gfu_old.vec.data = gfu.vec   # Stokes-Startwert
# #scene = Draw (gfu.components[0], mesh, min=0, max=2, autoscale=False)
# # ------------------------------------------------------------
# # EXPLIZITERES VERFAHREN MIT GETRENNTEN VECTORS
# # ------------------------------------------------------------

# # ... (Vorbereitung wie oben) ...

# # Explizite Berechnung der rechten Seite
# def ComputeRHS(u_old_vec):
#     """Berechne rechte Seite für gegebenen Vektor u_old."""
    
#     # 1. Zeitanteil
#     rhs = (1/dt) * (m.mat * u_old_vec)
    
#     # 2. Konvektionsanteil
#     # Temporäres GridFunction für Konvektion
#     tmp_gfu = GridFunction(X)
#     tmp_gfu.vec.data = u_old_vec
    
#     # Apply konvektion auf temporäres GF
#     conv_term = conv_form.Apply(tmp_gfu.vec)
    
#     rhs += conv_term
#     return rhs

# step = 0
# # Hauptschleife
# while t < t_end:
    
#     # 1. Rechte Seite mit ALTEM Zustand
#     rhs = ComputeRHS(gfu_old.vec)
    
#     # 2. Lösen für NEUEN Zustand
#     gfu.vec.data = invA_dt * rhs
    
#     # 3. Druck eindeutig machen
#     p_array = gfu.components[1].vec.FV().NumPy()
#     p_array[:] = p_array[:] - np.mean(p_array)
    
#     # 4. Überwachung
#     # if step % 10 == 0:
#     #     # Energie der Lösung
#     #     energy = InnerProduct(gfu.vec, m.mat * gfu.vec)
#     #     print(f"t={t:.3f}, Energie={energy:.6f}")
    
#     # 5. Update
#     gfu_old.vec.data = gfu.vec
#     t += dt
#     step += 1

# Draw(gfu.components[0],mesh)
# Draw(gfu_old.components[0],mesh)'

dt = 0.05
t_end = 50.0
t = 0.0

tau = 0.7          # 1.0 = ein Schritt ist exakt; <1.0 = gedämpft (z.B. 0.7)

# -----------------------------
# Massenmatrix (auf X; Druckblock ist dabei automatisch 0)
# -----------------------------
m = BilinearForm(X)
m += InnerProduct(u, v) * dx
m.Assemble()

# -----------------------------
# A_dt = (1/dt) M + Stokes (konstant)  -> einmal assemble + inverse
# -----------------------------
A_dt = BilinearForm(X)
A_dt += (1/dt) * InnerProduct(u, v) * dx
A_dt += nu * InnerProduct(Grad(u), Grad(v)) * dx
A_dt += gamma * div(u)*div(v) * dx
A_dt += -div(v)*p * dx
A_dt += -div(u)*q * dx
A_dt += alpha * p*q * dx
A_dt.Assemble()

invA_dt = A_dt.mat.Inverse(X.FreeDofs())

# -----------------------------
# Konvektion als LinearForm (außerhalb!), abhängig von uconv
# conv_vec = ∫ ( (uconv·∇)uconv ) · v  dx   (hier in NGSolve: Grad(uconv)*uconv)
# -----------------------------
uconv = GridFunction(V)     # hält jeweils u^n
conv = LinearForm(X)
conv += InnerProduct(Grad(uconv) * uconv, v) * dx   # Vektorwertig, getestet gegen v

# Startwert aus Stokes:
gfu_old = GridFunction(X)
gfu_old.vec.data = gfu.vec

# Hilfsvektoren einmal anlegen (Performance)
rhs = gfu.vec.CreateVector()
res = gfu.vec.CreateVector()

step = 0
while t < t_end:

    # uconv = u^n
    uconv.vec.data = gfu_old.components[0].vec

    # conv(u^n) auswerten (nur Assemble, Definition blieb außerhalb)
    conv.Assemble()
    conv_vec = conv.vec

    # rhs = (1/dt) * M * gfu_old
    rhs.data = (1/dt) * (m.mat * gfu_old.vec)

    # Residuum: res = A_dt*gfu + conv(u^n) - rhs
    res.data = (A_dt.mat * gfu.vec) + conv_vec - rhs

    # Update: gfu <- gfu - tau * invA_dt * res
    gfu.vec.data -= tau * (invA_dt * res)

    # Update für nächsten Zeitschritt
    gfu_old.vec.data = gfu.vec

    t += dt
    step += 1
    if step % 10 == 0:
        divnorm = sqrt(Integrate(div(gfu.components[0])**2, mesh))
        #print(f"t = {t:.3f}, div(u) = {divnorm:.2e}")

Draw(gfu.components[0], mesh, "Velocity")
Draw(gfu.components[1], mesh, "Pressure")


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene